<a href="https://colab.research.google.com/github/Iamjohnko/Data-science-Project-Portfolio/blob/main/Integrated_Petroleum_Reserves%2C_Production_Forecasting_%26_Economic_Valuation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Overview and Objective

This notebook analyzes petroleum reserves using decline curve analysis and economic modeling. The objective is to:

1.  **Estimate EUR (Estimated Ultimate Recovery)**: Apply Arps' decline curve equations to calculate EUR for individual wells.
2.  **Perform Economic Evaluation**: Build a cash flow model to calculate NPV (Net Present Value) at the well, field, and portfolio levels.
3.  **Conduct Risk Analysis**: Utilize Monte Carlo simulations to estimate probabilistic reserves (1P, 2P, 3P) and perform scenario analysis for different oil prices.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. LOAD DATA
# ==============================
df = pd.read_csv("petroleum_reserves_dataset.csv")

# Ensure correct ordering
df = df.sort_values(by=["Field", "Well", "Year"])

In [ ]:
df.head(10)

,Field,Well,Year,Rate_bbl_per_day,Annual_Production_bbl,qi,di,b_factor
0,Field_1,F1_W1,2010,2185.430535,797682.145207,2185.430535,0.240143,1.124792
1,Field_1,F1_W1,2011,1766.917574,644924.914439,2185.430535,0.240143,1.124792
2,Field_1,F1_W1,2012,1488.557255,543323.398112,2185.430535,0.240143,1.124792
3,Field_1,F1_W1,2013,1289.365729,470618.491139,2185.430535,0.240143,1.124792
4,Field_1,F1_W1,2014,1139.408580,415884.131602,2185.430535,0.240143,1.124792
5,Field_1,F1_W1,2015,1022.225895,373112.451749,2185.430535,0.240143,1.124792
6,Field_1,F1_W1,2016,927.997676,338719.151735,2185.430535,0.240143,1.124792
7,Field_1,F1_W1,2017,850.492774,310429.862546,2185.430535,0.240143,1.124792
8,Field_1,F1_W1,2018,785.561582,286729.977362,2185.430535,0.240143,1.124792
9,Field_1,F1_W1,2019,730.330989,266570.811025,2185.430535,0.240143,1.124792


In [ ]:
# 2. DECLINE CURVE (ARPS EUR)
# ==============================

def calculate_eur(qi, di, b, t_limit=20):
    """
    EUR for hyperbolic decline (years converted to days)
    """
    t = np.linspace(0, t_limit, 1000)

    if b == 0:
        q = qi * np.exp(-di * t)
    else:
        q = qi / ((1 + b * di * t) ** (1 / b))

    eur = np.trapz(q * 365, t)  # integrate annual production
    return eur

# Compute EUR per well
well_params = df.groupby("Well").first().reset_index()

well_params["EUR_bbl"] = well_params.apply(
    lambda x: calculate_eur(x["qi"], x["di"], x["b_factor"]),
    axis=1
)

/tmp/ipykernel_9753/3960018489.py:15: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  eur = np.trapz(q * 365, t)  # integrate annual production


In [ ]:
# ==============================
# 3. PRODUCTION AGGREGATION
# ==============================

# Field-level production
field_prod = df.groupby(["Field", "Year"])["Annual_Production_bbl"].sum().reset_index()

# Portfolio-level production
portfolio_prod = df.groupby("Year")["Annual_Production_bbl"].sum().reset_index()


In [ ]:
# ==============================
# 4. ECONOMIC MODEL ASSUMPTIONS
# ==============================

oil_price = 70  # $/bbl
lifting_cost = 15  # $/bbl
capex_per_well = 5_000_000  # $
discount_rate = 0.1

# Assign CAPEX at first production year
first_year = df.groupby("Well")["Year"].min().reset_index()
first_year["CAPEX"] = capex_per_well

df = df.merge(first_year, on=["Well", "Year"], how="left")
df["CAPEX"] = df["CAPEX"].fillna(0)

In [ ]:
# ==============================
# 5. CASH FLOW MODEL
# ==============================

df["Revenue"] = df["Annual_Production_bbl"] * oil_price
df["OPEX"] = df["Annual_Production_bbl"] * lifting_cost

df["CashFlow"] = df["Revenue"] - df["OPEX"] - df["CAPEX"]

# Discount factor
df["Year_Index"] = df["Year"] - df["Year"].min()
df["Discount_Factor"] = (1 + discount_rate) ** df["Year_Index"]

df["Discounted_CF"] = df["CashFlow"] / df["Discount_Factor"]

In [ ]:
# ==============================
# 6. NPV CALCULATIONS
# ==============================

# Well-level NPV
well_npv = df.groupby("Well")["Discounted_CF"].sum().reset_index()

# Field-level NPV
field_npv = df.groupby("Field")["Discounted_CF"].sum().reset_index()

# Portfolio NPV
portfolio_npv = df["Discounted_CF"].sum()

In [ ]:
display(well_npv)

,Well,Discounted_CF
0,F1_W1,2.071958e+08
1,F1_W10,2.501590e+08
2,F1_W11,3.912383e+08
3,F1_W12,4.650541e+08
4,F1_W13,2.551918e+08
...,...,...
95,F5_W5,4.802440e+08
96,F5_W6,2.853786e+08
97,F5_W7,4.190622e+08
98,F5_W8,6.550391e+08


In [ ]:
display(field_npv)

,Field,Discounted_CF
0,Field_1,5.902435e+09
1,Field_2,4.574630e+09
2,Field_3,5.462545e+09
3,Field_4,5.915196e+09
4,Field_5,6.772067e+09


In [ ]:
# 7. MONTE CARLO RESERVES (1P/2P/3P)
# ==============================

def monte_carlo_eur(qi, di, b, n_sim=1000):
    results = []

    for _ in range(n_sim):
        qi_sim = qi * np.random.normal(1, 0.2)
        di_sim = di * np.random.normal(1, 0.15)
        b_sim = b * np.random.normal(1, 0.1)

        eur = calculate_eur(qi_sim, di_sim, b_sim)
        results.append(eur)

    return np.percentile(results, [10, 50, 90])  # P10, P50, P90

reserves_mc = well_params.copy()

reserves_mc[["P90_1P", "P50_2P", "P10_3P"]] = reserves_mc.apply(
    lambda x: pd.Series(monte_carlo_eur(x["qi"], x["di"], x["b_factor"])),
    axis=1
)


/tmp/ipykernel_9753/3960018489.py:15: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  eur = np.trapz(q * 365, t)  # integrate annual production


In [ ]:
display(reserves_mc.head())

,Well,Field,Year,Rate_bbl_per_day,Annual_Production_bbl,qi,di,b_factor,EUR_bbl,P90_1P,P50_2P,P10_3P
0,F1_W1,Field_1,2010,2185.430535,7.976821e+05,2185.430535,0.240143,1.124792,6.088557e+06,4.506790e+06,6.057873e+06,7.841794e+06
1,F1_W10,Field_1,2010,2814.054973,1.027130e+06,2814.054973,0.168483,0.165031,6.521827e+06,4.634430e+06,6.584554e+06,8.639237e+06
2,F1_W11,Field_1,2010,3233.951834,1.180392e+06,3233.951834,0.084105,0.191072,1.201798e+07,8.837936e+06,1.188922e+07,1.536676e+07
3,F1_W12,Field_1,2010,4769.984918,1.741044e+06,4769.984918,0.243126,1.231756,1.364898e+07,1.002311e+07,1.351001e+07,1.770714e+07
4,F1_W13,Field_1,2010,1870.761961,6.828281e+05,1870.761961,0.069534,1.057926,8.609401e+06,6.465858e+06,8.747268e+06,1.109533e+07


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# ==============================
# 8. SCENARIO ANALYSIS
# ==============================

def run_price_scenario(price):
    df_scn = df.copy()

    df_scn["Revenue"] = df_scn["Annual_Production_bbl"] * price
    df_scn["CashFlow"] = df_scn["Revenue"] - df_scn["OPEX"] - df_scn["CAPEX"]
    df_scn["Discounted_CF"] = df_scn["CashFlow"] / df_scn["Discount_Factor"]

    return df_scn["Discounted_CF"].sum()

price_scenarios = {
    "Low_50": run_price_scenario(50),
    "Base_70": run_price_scenario(70),
    "High_90": run_price_scenario(90)
}

In [ ]:
display(price_scenarios)

{'Low_50': np.float64(18035282588.495876),
 'Base_70': np.float64(28626872639.06495),
 'High_90': np.float64(39218462689.63402)}

In [ ]:
# ==============================
# 9. OUTPUT SUMMARY
# ==============================

print("\n=== PORTFOLIO NPV ===")
print(portfolio_npv)

print("\n=== FIELD NPV ===")
print(field_npv)

print("\n=== PRICE SCENARIOS ===")
print(price_scenarios)

print("\n=== RESERVES (Sample) ===")
print(reserves_mc[["Well", "P90_1P", "P50_2P", "P10_3P"]].head())


=== PORTFOLIO NPV ===
28626872639.06495

=== FIELD NPV ===
     Field  Discounted_CF
0  Field_1   5.902435e+09
1  Field_2   4.574630e+09
2  Field_3   5.462545e+09
3  Field_4   5.915196e+09
4  Field_5   6.772067e+09

=== PRICE SCENARIOS ===
{'Low_50': np.float64(18035282588.495876), 'Base_70': np.float64(28626872639.06495), 'High_90': np.float64(39218462689.63402)}

=== RESERVES (Sample) ===
     Well        P90_1P        P50_2P        P10_3P
0   F1_W1  4.506790e+06  6.057873e+06  7.841794e+06
1  F1_W10  4.634430e+06  6.584554e+06  8.639237e+06
2  F1_W11  8.837936e+06  1.188922e+07  1.536676e+07
3  F1_W12  1.002311e+07  1.351001e+07  1.770714e+07
4  F1_W13  6.465858e+06  8.747268e+06  1.109533e+07
